# 🌿 MangroveNet v2 — Improved Training Notebook

**Key architectural improvements in this version:**
1. **UNet-style skip connections** — preserves spatial detail from ResNet layers 1-3
2. **Spatial transformer output** — returns attention-refined features instead of a single CLS vector
3. **1×1 bottleneck** — reduces 4096→1024 channels, cutting ~17M parameters
4. **Optimized training recipe** — longer freeze, lower backbone LR, Dice-heavy loss

**Expected improvement:** IoU 0.47 → 0.60–0.70+

---
> ⚠️ **Prerequisites:** Upload `dataset_patches.zip` to your Google Drive root (`My Drive/dataset_patches.zip`) before running.

## 1. Check GPU & Environment

In [ ]:
import torch
import subprocess

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU (T4)")
    raise RuntimeError("GPU required for training")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

## 2. Setup Environment

In [ ]:
import os

REPO_URL = "https://github.com/prajwal-gowd/magrove-mapping.git"
REPO_DIR = "/content/magrove-mapping"

if os.path.exists(REPO_DIR):
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Install dependencies (minimal for training)
!pip install -q torch torchvision tqdm matplotlib seaborn scikit-learn Pillow numpy
print("\n✅ Environment ready")

## 3. Mount Google Drive & Setup Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

# Look for dataset zip in common locations
zip_candidates = [
    "/content/drive/MyDrive/dataset_patches.zip",
    "/content/drive/MyDrive/mangrove_checkpoints/dataset_patches.zip",
    "/content/drive/My Drive/dataset_patches.zip",
]

zip_path = None
for candidate in zip_candidates:
    if os.path.exists(candidate):
        zip_path = candidate
        break

if zip_path is None:
    print("❌ dataset_patches.zip not found in Google Drive!")
    print("   Please upload it to: My Drive/dataset_patches.zip")
    print("   Searched locations:", zip_candidates)
    raise FileNotFoundError("Dataset not found")

print(f"Found dataset at: {zip_path}")

# Extract dataset
if not os.path.exists("dataset_patches/train/images"):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(".")
    print("✅ Dataset extracted")
else:
    print("✅ Dataset already extracted")

# Verify dataset
train_imgs = len(os.listdir("dataset_patches/train/images"))
train_masks = len(os.listdir("dataset_patches/train/masks"))
val_imgs = len(os.listdir("dataset_patches/val/images"))
val_masks = len(os.listdir("dataset_patches/val/masks"))
print(f"\nDataset summary:")
print(f"  Train: {train_imgs} images, {train_masks} masks")
print(f"  Val:   {val_imgs} images, {val_masks} masks")

## 4. Apply Architecture Improvements

The following cells overwrite the model files with the improved v2 architecture:
- **Skip connections** from ResNet layers 1/2/3 to decoder
- **Spatial transformer output** instead of CLS-only
- **1×1 bottleneck** to reduce parameters

In [ ]:
%%writefile models/resnet_backbone.py
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights


class ResNetBackbone(nn.Module):
    """
    ResNet-50 backbone with multi-scale feature extraction for UNet-style
    skip connections.
    """

    def __init__(self, pretrained=True):
        super(ResNetBackbone, self).__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        resnet = resnet50(weights=weights)

        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
        )
        self.layer1 = resnet.layer1  # -> [B, 256, 56, 56]
        self.layer2 = resnet.layer2  # -> [B, 512, 28, 28]
        self.layer3 = resnet.layer3  # -> [B, 1024, 14, 14]
        self.layer4 = resnet.layer4  # -> [B, 2048, 7, 7]
        self.avgpool = resnet.avgpool

    def forward(self, x):
        x = self.stem(x)
        s1 = self.layer1(x)    # [B, 256, 56, 56]
        s2 = self.layer2(s1)   # [B, 512, 28, 28]
        s3 = self.layer3(s2)   # [B, 1024, 14, 14]
        s4 = self.layer4(s3)   # [B, 2048, 7, 7]
        global_features = torch.flatten(self.avgpool(s4), 1)
        return s4, global_features, [s1, s2, s3]

In [ ]:
%%writefile models/transformer_encoder.py
import torch
import torch.nn as nn


class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, height, width):
        super(PositionalEncoding2D, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, height * width, d_model))

    def forward(self, x):
        return x + self.pos_embedding


class TransformerModule(nn.Module):
    """
    Vision Transformer that returns spatially-refined features
    instead of only a CLS token.
    """

    def __init__(self, d_model=2048, num_heads=8, num_layers=2,
                 dim_feedforward=512, dropout=0.1, feat_h=7, feat_w=7):
        super(TransformerModule, self).__init__()
        self.d_model = d_model
        self.feat_h = feat_h
        self.feat_w = feat_w
        self.pos_encoder = PositionalEncoding2D(d_model, feat_h, feat_w)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)        # [B, H*W, C]
        x = self.pos_encoder(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)          # [B, 1+H*W, C]
        x = self.transformer_encoder(x)
        spatial_out = x[:, 1:, :]                       # [B, H*W, C]
        spatial_out = spatial_out.permute(0, 2, 1).view(B, C, H, W)
        return spatial_out

In [ ]:
%%writefile models/hybrid_model.py
import torch
import torch.nn as nn
from .resnet_backbone import ResNetBackbone
from .transformer_encoder import TransformerModule
import config


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class HybridTCCFNet(nn.Module):
    """
    Improved HybridTCCFNet with UNet-style skip connections,
    spatial transformer output, and 1x1 bottleneck.
    """

    def __init__(self, num_classes=1, backbone_freeze=False, pretrained_backbone=True):
        super(HybridTCCFNet, self).__init__()
        self.cnn = ResNetBackbone(pretrained=pretrained_backbone)
        if backbone_freeze:
            for param in self.cnn.parameters():
                param.requires_grad = False

        self.transformer = TransformerModule(
            d_model=2048, num_heads=config.TRANSFORMER_HEADS,
            num_layers=config.TRANSFORMER_LAYERS,
            dim_feedforward=config.TRANSFORMER_DIM_FEEDFORWARD,
            dropout=config.DROPOUT, feat_h=7, feat_w=7,
        )

        dec_drop = getattr(config, "DECODER_DROPOUT", 0.2)

        # Bottleneck: 4096 -> 1024
        self.bottleneck = nn.Sequential(
            nn.Conv2d(4096, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024), nn.ReLU(inplace=True),
        )

        # UNet Decoder with skip connections
        self.up3 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec3 = DecoderBlock(512 + 1024, 512, dropout=dec_drop)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DecoderBlock(256 + 512, 256, dropout=dec_drop)

        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = DecoderBlock(128 + 256, 128, dropout=dec_drop)

        self.up0 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec0 = DecoderBlock(64, 64)

        self.final_up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def freeze_backbone(self):
        for param in self.cnn.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.cnn.parameters():
            param.requires_grad = True

    def get_param_groups(self, backbone_lr, decoder_lr):
        backbone_params = list(self.cnn.parameters())
        backbone_ids = set(id(p) for p in backbone_params)
        other_params = [p for p in self.parameters() if id(p) not in backbone_ids]
        return [
            {"params": backbone_params, "lr": backbone_lr},
            {"params": other_params, "lr": decoder_lr},
        ]

    def forward(self, x):
        x4, _, skips = self.cnn(x)
        skip1, skip2, skip3 = skips

        transformer_features = self.transformer(x4)
        fused = torch.cat([x4, transformer_features], dim=1)
        fused = self.bottleneck(fused)

        d3 = self.up3(fused)
        d3 = torch.cat([d3, skip3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, skip2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, skip1], dim=1)
        d1 = self.dec1(d1)

        d0 = self.up0(d1)
        d0 = self.dec0(d0)

        out = self.final_up(d0)
        logits = self.head(out)
        return logits

## 5. Verify Improved Architecture

In [ ]:
import torch
import importlib, sys

# Force reimport to pick up overwritten files
for mod_name in list(sys.modules.keys()):
    if 'models' in mod_name or 'config' in mod_name:
        del sys.modules[mod_name]

import config
from models.hybrid_model import HybridTCCFNet

print("=== HybridTCCFNet v2 Architecture Verification ===\n")
model = HybridTCCFNet(num_classes=1)
model.eval()

dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model(dummy)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"\nParam reduction from v1: {97_739_777 - total:,} fewer parameters")

assert out.shape == (2, 1, 224, 224), f"Wrong shape: {out.shape}"
print("\n✅ Architecture verified!")
del model, dummy, out
torch.cuda.empty_cache()

## 6. Train the Model

**Optimized hyperparameters:**
| Parameter | Value | Rationale |
|:---|:---:|:---|
| `batch-size` | 16 | Fills T4 GPU memory efficiently |
| `lr` (decoder) | 5e-4 | Decoder trains from scratch, needs faster convergence |
| `backbone-lr` | 1e-5 | Prevents destroying pretrained ImageNet features |
| `freeze-epochs` | 10 | Lets decoder converge before fine-tuning backbone |
| `warmup-epochs` | 5 | Smooth warmup for training stability |
| `bce-weight / dice-weight` | 0.3 / 0.7 | Dice-heavy loss directly optimizes IoU |
| `threshold` | 0.5 | Start neutral, optimize post-training |
| `epochs` | 80 | Enough for cosine schedule convergence |

In [ ]:
import config

# Configure the experiment through config.py values so this works with
# repository versions whose train.py only accepts --resume.
config.BATCH_SIZE = 16
config.LEARNING_RATE = 5e-4
config.BACKBONE_LR = 1e-5
config.NUM_EPOCHS = 80
config.WARMUP_EPOCHS = 5
config.BACKBONE_FREEZE_EPOCHS = 10
config.VAL_THRESHOLD = 0.5
config.BCE_WEIGHT = 0.3
config.DICE_WEIGHT = 0.7
config.FOCAL_ALPHA = 0.5
config.FOCAL_GAMMA = 2.0
config.SAVE_PATH = "checkpoints/best_model_v2.pth"

# Calling train() directly avoids relying on optional CLI flags.
from train import train
train()


## 7. Find Optimal Decision Threshold

In [ ]:
import os
import torch
import numpy as np
import sys

# Reimport modules
for mod_name in list(sys.modules.keys()):
    if 'models' in mod_name or 'config' in mod_name:
        del sys.modules[mod_name]

import config
from models.hybrid_model import HybridTCCFNet
from utils.dataloader import get_dataloaders
from utils.metrics import calculate_iou, calculate_dice, calculate_precision, calculate_recall

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model
model = HybridTCCFNet(num_classes=1, pretrained_backbone=False)
ckpt_path = "checkpoints/best_model_v2.pth"
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f"Checkpoint not found: {ckpt_path}. Run the training cell successfully first."
    )
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
    best_epoch = checkpoint.get("epoch", "?")
    best_iou = checkpoint.get("best_val_iou", "?")
    print(f"Loaded checkpoint from epoch {best_epoch}, training IoU: {best_iou:.4f}")
else:
    model.load_state_dict(checkpoint)
model.to(device)
model.eval()

_, val_loader = get_dataloaders(batch_size=16)

# Sweep thresholds
thresholds = np.arange(0.30, 0.85, 0.05)
results = []

print(f"\n{'Threshold':>10} {'IoU':>8} {'Dice':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 52)

for thresh in thresholds:
    total_iou = total_dice = total_prec = total_rec = 0
    total_samples = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > thresh).float()

            total_iou += calculate_iou(preds, masks) * images.size(0)
            total_dice += calculate_dice(preds, masks) * images.size(0)
            total_prec += calculate_precision(preds, masks) * images.size(0)
            total_rec += calculate_recall(preds, masks) * images.size(0)
            total_samples += images.size(0)

    avg_iou = total_iou / total_samples
    avg_dice = total_dice / total_samples
    avg_prec = total_prec / total_samples
    avg_rec = total_rec / total_samples
    results.append((thresh, avg_iou, avg_dice, avg_prec, avg_rec))
    print(f"{thresh:>10.2f} {avg_iou:>8.4f} {avg_dice:>8.4f} {avg_prec:>10.4f} {avg_rec:>8.4f}")

# Find best threshold
best = max(results, key=lambda x: x[1])
print(f"\n✅ Best threshold: {best[0]:.2f} → IoU={best[1]:.4f}, Dice={best[2]:.4f}")
print(f"   Precision={best[3]:.4f}, Recall={best[4]:.4f}")

BEST_THRESHOLD = best[0]

## 8. Generate Confusion Matrix & Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from utils.metrics import calculate_confusion_matrix_elements, calculate_accuracy

# Evaluate with best threshold
total_tp = total_tn = total_fp = total_fn = 0
total_acc = 0
total_samples = 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > BEST_THRESHOLD).float()

        tp, tn, fp, fn = calculate_confusion_matrix_elements(preds, masks)
        total_tp += tp
        total_tn += tn
        total_fp += fp
        total_fn += fn
        total_acc += calculate_accuracy(preds, masks) * images.size(0)
        total_samples += images.size(0)

avg_acc = total_acc / total_samples
iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
dice = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6)
precision = total_tp / (total_tp + total_fp + 1e-6)
recall = total_tp / (total_tp + total_fn + 1e-6)

print("=" * 50)
print(f"  FINAL RESULTS (threshold={BEST_THRESHOLD:.2f})")
print("=" * 50)
print(f"  IoU:       {iou:.4f}")
print(f"  Dice/F1:   {dice:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Accuracy:  {avg_acc:.4f}")
print(f"  TP={int(total_tp):,}  TN={int(total_tn):,}")
print(f"  FP={int(total_fp):,}  FN={int(total_fn):,}")
print("=" * 50)

# Confusion Matrix Plot
cm = np.array([[total_tn, total_fp], [total_fn, total_tp]])
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = cm / row_sums * 100

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
labels = np.array([
    [f"TN\n{int(total_tn):,}\n({cm_norm[0,0]:.2f}%)", f"FP\n{int(total_fp):,}\n({cm_norm[0,1]:.2f}%)"],
    [f"FN\n{int(total_fn):,}\n({cm_norm[1,0]:.2f}%)", f"TP\n{int(total_tp):,}\n({cm_norm[1,1]:.2f}%)"],
])
sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", ax=ax,
            xticklabels=["Non-Mangrove (0)", "Mangrove (1)"],
            yticklabels=["Non-Mangrove (0)", "Mangrove (1)"])
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title(f"Confusion Matrix — v2 Model (IoU: {iou:.4f}, Threshold: {BEST_THRESHOLD:.2f})")
plt.tight_layout()
plt.savefig("confusion_matrix_v2.png", dpi=300)
plt.show()
print("Saved confusion_matrix_v2.png")

## 9. Visual Predictions on Validation Samples

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

# Get a few validation samples
val_dir = "dataset_patches/val/images"
mask_dir = "dataset_patches/val/masks"
sample_names = sorted(os.listdir(val_dir))[:6]

fig, axes = plt.subplots(len(sample_names), 3, figsize=(12, 4 * len(sample_names)))
if len(sample_names) == 1:
    axes = axes.reshape(1, -1)

for i, name in enumerate(sample_names):
    img = Image.open(os.path.join(val_dir, name)).convert("RGB")
    gt_mask = Image.open(os.path.join(mask_dir, name)).convert("L")

    # Preprocess
    img_tensor = TF.resize(img, (224, 224))
    img_tensor = TF.to_tensor(img_tensor)
    img_tensor = TF.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()

    pred_mask = (pred > BEST_THRESHOLD).astype(np.uint8) * 255

    axes[i, 0].imshow(img.resize((224, 224)))
    axes[i, 0].set_title(f"Input: {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(np.array(gt_mask.resize((224, 224))), cmap="gray")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred_mask, cmap="gray")
    axes[i, 2].set_title(f"Prediction (t={BEST_THRESHOLD:.2f})")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("visual_predictions_v2.png", dpi=150)
plt.show()
print("Saved visual_predictions_v2.png")

## 10. v1 vs v2 Performance Comparison

In [ ]:
# v1 baseline metrics
v1 = {"IoU": 0.4716, "Dice": 0.6505, "Precision": 0.5192, "Recall": 0.8706, "Accuracy": 0.5710}
v2 = {"IoU": iou, "Dice": dice, "Precision": precision, "Recall": recall, "Accuracy": avg_acc}

metrics = list(v1.keys())
v1_vals = [v1[m] for m in metrics]
v2_vals = [v2[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, v1_vals, width, label="v1 (baseline)", color="#ef4444", alpha=0.8)
bars2 = ax.bar(x + width/2, v2_vals, width, label="v2 (improved)", color="#22c55e", alpha=0.8)

ax.set_ylabel("Score")
ax.set_title("MangroveNet: v1 vs v2 Performance")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("v1_vs_v2_comparison.png", dpi=150)
plt.show()

print(f"\n{'Metric':<12} {'v1':>8} {'v2':>8} {'Change':>10}")
print("-" * 40)
for m in metrics:
    change = v2[m] - v1[m]
    arrow = "↑" if change > 0 else "↓" if change < 0 else "="
    print(f"{m:<12} {v1[m]:>8.4f} {v2[m]:>8.4f} {arrow} {abs(change):>+8.4f}")

## 11. Save Results to Google Drive

In [ ]:
import shutil

save_dir = "/content/drive/MyDrive/mangrove_checkpoints"
os.makedirs(save_dir, exist_ok=True)

files_to_save = [
    ("checkpoints/best_model_v2.pth", "best_model_v2.pth"),
    ("training_v2.log", "training_v2.log"),
    ("confusion_matrix_v2.png", "confusion_matrix_v2.png"),
    ("visual_predictions_v2.png", "visual_predictions_v2.png"),
    ("v1_vs_v2_comparison.png", "v1_vs_v2_comparison.png"),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(save_dir, dst))
        print(f"✅ Saved: {dst}")
    else:
        print(f"⚠️  Not found: {src}")

# Also save the best threshold
with open(os.path.join(save_dir, "best_threshold.txt"), "w") as f:
    f.write(f"Best threshold: {BEST_THRESHOLD:.4f}\n")
    f.write(f"IoU: {iou:.4f}\n")
    f.write(f"Dice: {dice:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")

print(f"\n✅ All results saved to: {save_dir}")
print(f"\n📋 Summary:")
print(f"   Best threshold: {BEST_THRESHOLD:.2f}")
print(f"   IoU:  {iou:.4f}  (v1: 0.4716)")
print(f"   Dice: {dice:.4f}  (v1: 0.6505)")